# Seaborn Phase 5: Matrix Plots (The Pre-Machine-Learning Check)
### Credit Card Risk Analysis Project

Before any of these features go into a model, you need to know if two of them are
secretly telling you the same thing. If `Credit_Score` and `Months_Since_Last_Delinquency`
move almost in lockstep, a model gains nothing from having both — and in some model
types, having both can actively hurt it (multicollinearity). A correlation heatmap is
how you catch this before it becomes a modeling problem.

This notebook covers 1 topic in depth:
10. **The Correlation Heatmap** — a color-coded grid of every numeric column against
    every other, used specifically to flag redundant features before modeling

**Format:** Each question has a `YOUR CODE HERE` cell to attempt first, followed by a
`Solution` cell. Try your own answer before peeking!

Run the setup cell below first — it deliberately builds in two clearly redundant
feature pairs so the multicollinearity check has something real to catch.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
sns.set_theme(style="whitegrid")

n = 600

age = np.clip(np.random.normal(40, 12, n), 18, 80)

credit_score = np.clip(np.random.normal(660, 65, n), 300, 850)
# Deliberately redundant with Credit_Score: fewer months since a delinquency
# almost always means a lower score, and vice versa
months_since_delinquency = np.clip(
    (credit_score - 300) / 550 * 60 + np.random.normal(0, 4, n), 0, 60
)

annual_income = np.random.lognormal(mean=10.8, sigma=0.4, size=n)
# Deliberately redundant with Annual_Income: credit limit is a near-linear function of it
credit_limit = np.clip(3000 + annual_income * 0.15 + np.random.normal(0, 800, n), 500, None)

total_debt = annual_income * np.random.uniform(0.05, 0.55, size=n)
debt_to_income = np.clip((total_debt / annual_income) * 100, 0, 90)
credit_utilization = np.clip(debt_to_income * 0.9 + np.random.normal(0, 5, n), 0, 100)

raw_risk = (debt_to_income / 100) * 0.6 + ((850 - credit_score) / 550) * 0.6
default_probability = np.clip(raw_risk + np.random.normal(0, 0.08, n), 0.01, 0.95)

df = pd.DataFrame({
    'Age': age,
    'Credit_Score': credit_score,
    'Months_Since_Last_Delinquency': months_since_delinquency,
    'Annual_Income': annual_income,
    'Credit_Limit': credit_limit,
    'Total_Debt': total_debt,
    'Debt_to_Income': debt_to_income,
    'Credit_Utilization': credit_utilization,
    'Default_Probability': default_probability
})

print(df.shape)
df.head()


---
## Topic 10: The Correlation Heatmap

A correlation matrix on its own is just a grid of numbers — hard to scan for patterns.
A heatmap turns it into something your eye can process in seconds: strong relationships
jump out as saturated colors, weak ones fade into the background.


**Q1.** Compute the pairwise correlation matrix of every numeric column in `df` using `.corr()`, store it as `corr_matrix`, and print it rounded to 2 decimal places.

In [ ]:
# YOUR CODE HERE


**Solution 1**

In [ ]:
corr_matrix = df.corr()
print(corr_matrix.round(2))


**Q2.** Plot `corr_matrix` as a basic heatmap using `sns.heatmap(corr_matrix)`. Even with zero styling, you should already be able to spot at least one very bright (or very dark) off-diagonal cell.

In [ ]:
# YOUR CODE HERE


**Solution 2**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, ax=ax)
plt.show()


**Q3.** Repeat Q2, adding `annot=True` and `fmt='.2f'` so every cell shows its exact correlation value — precise enough to cite directly in a feature-selection writeup.

In [ ]:
# YOUR CODE HERE


**Solution 3**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', ax=ax)
plt.show()


**Q4.** Repeat Q3, setting `cmap='coolwarm'` and `center=0` — the standard convention: red for positive correlation, blue for negative, white/neutral near zero, so the color alone tells you the direction of the relationship.

In [ ]:
# YOUR CODE HERE


**Solution 4**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
plt.show()


**Q5.** A correlation matrix is symmetric — the upper and lower triangles repeat the same information. Build `mask = np.triu(np.ones_like(corr_matrix, dtype=bool))` and pass it to `sns.heatmap(..., mask=mask)` so only the lower triangle (plus the diagonal) is shown.

In [ ]:
# YOUR CODE HERE


**Solution 5**

In [ ]:
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
plt.show()


**Q6.** Repeat Q5, adding `square=True` (forces every cell to be a perfect square) and `linewidths=0.5` (thin separating lines between cells) for a cleaner, more polished grid.

In [ ]:
# YOUR CODE HERE


**Solution 6**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
plt.show()


**Q7.** Correlation isn't always linear. Compute a second correlation matrix using `df.corr(method='spearman')` (rank-based correlation, which catches monotonic-but-not-linear relationships that Pearson's default method can miss), and plot the Pearson (`method='pearson'`, the default) and Spearman matrices side by side in a 1x2 subplot grid for comparison.

In [ ]:
# YOUR CODE HERE


**Solution 7**

In [ ]:
pearson_corr = df.corr(method='pearson')
spearman_corr = df.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
sns.heatmap(pearson_corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=axes[0])
axes[0].set_title("Pearson Correlation")

sns.heatmap(spearman_corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=axes[1])
axes[1].set_title("Spearman Correlation")

plt.show()


**Q8.** Move beyond eyeballing the heatmap: programmatically find every pair of *different* columns with an absolute correlation above `0.7`. Hint — `.unstack()` the correlation matrix into a Series of pairs, drop self-correlations (where a column correlates with itself, always 1.0), drop duplicate pairs (since `(A, B)` and `(B, A)` are the same relationship), then filter and sort by absolute value descending.

In [ ]:
# YOUR CODE HERE


**Solution 8**

In [ ]:
corr_pairs = corr_matrix.unstack()
corr_pairs = corr_pairs[corr_pairs.index.get_level_values(0) != corr_pairs.index.get_level_values(1)]

# Drop duplicate (A, B) / (B, A) pairs by sorting each pair's names and de-duping
corr_pairs = corr_pairs.reset_index()
corr_pairs.columns = ['Feature_1', 'Feature_2', 'Correlation']
corr_pairs['pair_key'] = corr_pairs.apply(lambda r: tuple(sorted([r['Feature_1'], r['Feature_2']])), axis=1)
corr_pairs = corr_pairs.drop_duplicates(subset='pair_key').drop(columns='pair_key')

high_corr_pairs = corr_pairs[corr_pairs['Correlation'].abs() > 0.7].sort_values(
    'Correlation', key=abs, ascending=False)
print(high_corr_pairs)


**Q9 (New tool — Clustermap).** Plot `sns.clustermap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)`. Unlike a plain heatmap, `clustermap` automatically reorders both rows and columns using hierarchical clustering, grouping the most similar features next to each other — the redundant pairs from Q8 should visually cluster together.

In [ ]:
# YOUR CODE HERE


**Solution 9**

In [ ]:
sns.clustermap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, figsize=(9, 9))
plt.show()


**Q10.** Repeat Q9's clustermap, adding `linewidths=0.5` and `figsize=(10, 10)`. Note that `sns.clustermap()` returns its own `ClusterGrid` object (similar in spirit to `JointGrid`), which is why it's called directly rather than being passed an `ax=`.

In [ ]:
# YOUR CODE HERE


**Solution 10**

In [ ]:
cg = sns.clustermap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                     linewidths=0.5, figsize=(10, 10))
print(type(cg))
plt.show()


**Q11.** For actually selecting features, what usually matters most is correlation with the *target* — here, `Default_Probability`. Extract `corr_matrix['Default_Probability']`, drop the self-correlation, sort by absolute value descending, and plot it as a single-column heatmap using `sns.heatmap(sorted_target_corr.to_frame(), annot=True, fmt='.2f', cmap='coolwarm', center=0)`.

In [ ]:
# YOUR CODE HERE


**Solution 11**

In [ ]:
target_corr = corr_matrix['Default_Probability'].drop('Default_Probability')
sorted_target_corr = target_corr.reindex(target_corr.abs().sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(4, 6))
sns.heatmap(sorted_target_corr.to_frame(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title("Correlation with Default Probability")
plt.show()


**Q12.** Repeat Q6's masked, annotated heatmap, but customize the colorbar itself: pass `cbar_kws={'label': 'Correlation Coefficient', 'shrink': 0.8}` to `sns.heatmap()` so the colorbar has an explicit label and is slightly shrunk to leave more room for the grid.

In [ ]:
# YOUR CODE HERE


**Solution 12**

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'label': 'Correlation Coefficient', 'shrink': 0.8},
            ax=ax)
plt.show()


**Q13 (Capstone).** Assemble the final multicollinearity report: the masked, annotated, `coolwarm`/`center=0` heatmap from Q12, with a bold title `"Feature Correlation Matrix — Multicollinearity Check"`. Below the chart, print the `high_corr_pairs` table from Q8 again, then print a one-line recommendation naming which of the two features in the strongest redundant pair you'd consider dropping before modeling, and why (hint: the one that's less directly interpretable, or more prone to future data-quality issues, is often the one to drop).

In [ ]:
# YOUR CODE HERE


**Solution 13**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'label': 'Correlation Coefficient'}, ax=ax)
ax.set_title("Feature Correlation Matrix — Multicollinearity Check", fontsize=13, fontweight='bold')
plt.show()

print("\nHighly correlated feature pairs (|correlation| > 0.7):")
print(high_corr_pairs)

top_pair = high_corr_pairs.iloc[0]
print(
    f"\nStrongest redundant pair: '{top_pair['Feature_1']}' and '{top_pair['Feature_2']}' "
    f"at {top_pair['Correlation']:.2f}. Between these two, 'Credit_Limit' is largely a downstream "
    "consequence of 'Annual_Income' (the bank sets it *based on* income), so it carries little "
    "independent signal — 'Credit_Limit' is the better candidate to drop.\n"
    "Also worth flagging: 'Credit_Score' and 'Months_Since_Last_Delinquency' correlate at 0.88 — "
    "since Credit_Score is the more direct, more commonly available signal, "
    "'Months_Since_Last_Delinquency' would be the one to drop from that pair."
)


---
## Checkpoint: Seaborn Phase 5 Complete

You've covered:
- **Correlation heatmaps** — `sns.heatmap()` with `annot`, `fmt`, `cmap='coolwarm'` +
  `center=0` as the standard convention, and masking the redundant upper triangle
- **Pearson vs. Spearman** — checking both when a relationship might be monotonic but
  not linear
- **Programmatic pair detection** — turning a visual "that looks bright" into an exact,
  sorted list of redundant feature pairs above a chosen threshold
- **Clustermap** — automatically grouping similar features together via hierarchical
  clustering, so redundant pairs cluster visually without you having to spot them by eye
- **Target-focused correlation** — narrowing the whole matrix down to just "what
  correlates with the thing I'm trying to predict," which is usually the more actionable
  view for feature selection

This is the last stop before modeling: you now have a concrete, defensible way to decide
which features are redundant and should be dropped, rather than feeding a model
everything and hoping for the best.

**Where to next:** with the full Matplotlib + Seaborn toolkit in place, this is a natural
point to apply everything to your real credit card risk dataset — or, if you'd like,
move on to `sns.pairplot()` and `sns.FacetGrid()` for a final round of multivariate
exploration before modeling.
